# JN-A — Unconditional Ingestion: the CPRA BP feed → the v4 event stream

**This is the first brick of the v4 pipeline, and it has exactly one job:**
turn every row the city sent into an *event*, and **prove that nothing vanished.**

It does **no** classification, **no** housing judgement, **no** dedup, **no** date-window
filtering, **no** REV/UnitsAdded fixing. A row that looks like a temp-power permit, an
alteration, a duplicate, or an out-of-window date — *all* become events. Every judgement is
deferred to later notebooks as a **reversible label** on permanent evidence.

**Why so strict?** Every past data-loss in this project — the 568-unit drop, the 1173 Hearst
under-count, the Logan Park collapse — happened because a gate that should only have *labelled*
a row instead *deleted* it. v4 unwelds those: ingestion is unconditional, and the single place
data can still vanish (the ingest boundary) is guarded here by a conservation proof you can
re-run yourself.

**The "What Just Happened" sandwich.** Every code cell is preceded by a *What this cell does*
note and followed by a *What just happened* note. The first declares intent; the second
confronts the actual result. If the result contradicts the intent, that contradiction is the
finding — and the notebook is built to make it loud, not silent.

---

### Verified facts this notebook is built on (from read-only probes, 2026-06-25)
- The feed is two files: `BP_Annual Permit Report-2018-2022.xlsx` + `-2023-2025.xlsx`.
- **32,202 raw data rows** (pre-dedup). The deduped permit count is ~30,764 — so a correct
  ingest lands **well above 30,764 events**; anything near 30k means something deduped, which is
  a **bug at this stage**.
- Per-date-field non-null counts (our conservation anchors):
  **Submittal 32,202 · Issuance 31,940 · Finaled 21,650 · Completed 1.**
- The feed is **84% Alteration, 1,773 New** — a mixed permit feed, *not* "all BP". Those
  alterations must all become events and get *labelled* non-housing later, never filtered here.
- Completeness verified: every city BP-extract in the 26-1525 drop is a subset of this feed.
- Date formats are mixed (Submittal as datetime, Issuance as `MM/DD/YYYY` strings) → parse
  tolerantly and record `event_date_precision` honestly.

### The key design: one permit row is a *bundle* of lifecycle events, not one event
Each permit row carries up to four dated lifecycle fields. We **explode** each row into one
event per *present* date:

| date field present | event_type_code     | lifecycle phase | completion signal |
|--------------------|---------------------|-----------------|-------------------|
| Submittal Date     | `permit_submitted`  | BP_APPLIED      | no                |
| Issuance Date      | `permit_issued`     | BP_ISSUED       | no                |
| Finaled Date       | `permit_finaled`    | COMPLETION      | **yes**           |
| Completed Date     | `permit_completed`  | COMPLETION      | **yes**           |

A null date field emits **no** event for that field (we never invent dates). This is the *only*
typing JN-A does, and it is mechanical — driven purely by which date column is populated, never
by reading the description. Housing/master/role classification is JN-C's job, on intact evidence.

### What this cell does
Sets the paths and the known-truth anchors as named constants, so every downstream assertion
checks against a *stated expectation* rather than a magic number buried in code. If you are
running this in a different layout, this is the only cell you edit.

In [ ]:
from pathlib import Path
import sqlite3, json, datetime as dt
import pandas as pd

# --- paths (edit here if your layout differs) ---
REPO       = Path.home() / "berkeley-data"
SCHEMA_SQL = REPO / "schema" / "v4" / "schema_v4.sql"
DB_PATH    = REPO / "databases" / "berkeley_housing_v4.db"
FEED_DIR   = REPO / "data" / "raw" / "cpra-downloads"
FEED_GLOB  = "BP_Annual Permit Report-*.xlsx"

# --- known-truth anchors (from the read-only probes, 2026-06-25) ---
EXPECT_RAW_ROWS      = 32202          # total data rows across both files
EXPECT_DEDUP_FLOOR   = 30764          # event count must comfortably EXCEED this
EXPECT_SUBMITTAL     = 32202
EXPECT_ISSUANCE      = 31940
EXPECT_FINALED       = 21650
EXPECT_COMPLETED     = 1
REQUEST_WINDOW       = (dt.date(2018,1,1), dt.date(2025,12,31))

# --- the verbatim source column names (from the probe) ---
COL_SUBMITTAL = "Submittal Date"
COL_ISSUANCE  = "Issuance Date"
COL_FINALED   = "Finaled Date"
COL_COMPLETED = "Completed Date"
COL_PERMIT    = "PermitNumber"
HEADER_ROW    = 6   # 0-indexed: data headers are on the 7th row in these files

# the four lifecycle date fields -> (event_type_code, is_completion_signal)
DATE_EVENT_MAP = {
    COL_SUBMITTAL: ("permit_submitted", 0),
    COL_ISSUANCE:  ("permit_issued",    0),
    COL_FINALED:   ("permit_finaled",   1),
    COL_COMPLETED: ("permit_completed", 1),
}

NOW = dt.datetime.now(dt.timezone.utc).isoformat()
print("Config loaded.")
print("  DB target :", DB_PATH)
print("  Feed dir  :", FEED_DIR)
print("  Expecting :", EXPECT_RAW_ROWS, "raw rows ->  events must exceed", EXPECT_DEDUP_FLOOR)

### What just happened
The constants are now in memory. Nothing has been read or written yet. The anchors
(`EXPECT_*`) are the numbers the read-only probes established as ground truth — every
conservation check below compares against *these*, so a silent miscount can't pass as success.
If any path printed above is wrong for your machine, stop and fix this cell before continuing.

### What this cell does
Builds a **fresh** `berkeley_housing_v4.db` from the committed schema. We delete any existing v4
file first so this notebook is idempotent — re-running it always starts from a clean, schema-defined
database, never an accreted one. **v3 is never touched.** We then confirm the table count and that
foreign-key enforcement is on.

In [ ]:
# refuse to clobber anything that isn't our v4 target (paranoia: never delete v3)
assert DB_PATH.name == "berkeley_housing_v4.db", "Safety: refusing to build over a non-v4 file."

if DB_PATH.exists():
    DB_PATH.unlink()
    print("Removed prior v4 db (fresh rebuild).")

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON;")
con.executescript(SCHEMA_SQL.read_text())
con.commit()

tables = [r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
fk_problems = con.execute("PRAGMA foreign_key_check").fetchall()

print(f"Built {DB_PATH.name} from schema.")
print(f"  tables: {len(tables)}")
print(f"  FK integrity: {'CLEAN' if not fk_problems else fk_problems}")
assert len(tables) == 27, f"Expected 27 tables, got {len(tables)} - schema mismatch."
assert not fk_problems, "FK integrity problem in fresh schema."

### What just happened
A clean v4 database now exists with all 27 tables and no foreign-key violations — an empty,
schema-defined vessel. The two assertions are tripwires: if the table count isn't 27 or the FK
check isn't clean, the schema file on disk differs from what we expect, and we stop before
ingesting into a malformed database. v3 is untouched on disk.

### What this cell does
Seeds the **abstract, city-neutral** lifecycle vocabulary — the phases and the four event types
JN-A uses. This vocabulary is the contract every future city adapter targets, so it is defined in
the abstract (e.g. `COMPLETION`, evidenced-by a finaled permit) rather than in Berkeley's specific
terms. JN-A only needs the four `permit_*` types; later notebooks extend the vocabulary, they don't
replace it.

In [ ]:
phases = [
    # (phase_code, phase_order, assessment_half, description)
    ("CONCEPT",              10, "existing", "Pre-application concept/design (usually unsourced)"),
    ("ENTITLEMENT_APPLIED",  20, "existing", "Entitlement application submitted"),
    ("ENTITLEMENT_APPROVED", 30, "existing", "Land-use approval granted"),
    ("BP_APPLIED",           40, "existing", "Building permit submitted"),
    ("BP_ISSUED",            50, "existing", "Building permit issued; construction may begin"),
    ("CONSTRUCTION",         60, "existing", "Construction-phase inspections"),
    ("COMPLETION",           70, "existing", "Completion / certificate of occupancy / finaled"),
    ("TENURE",               80, "existing", "Sale / lease / occupancy (usually unsourced)"),
]
event_types = [
    # (event_type_code, phase_code, is_completion_signal, description)
    ("permit_submitted", "BP_APPLIED",  0, "BP application submittal (Submittal Date)"),
    ("permit_issued",    "BP_ISSUED",   0, "BP issuance (Issuance Date) - the RHNA-credit milestone"),
    ("permit_finaled",   "COMPLETION",  1, "BP finaled (Finaled Date) - Berkeley's CO-equivalent"),
    ("permit_completed", "COMPLETION",  1, "Completion recorded (Completed Date)"),
]
con.executemany("INSERT INTO lifecycle_phases VALUES (?,?,?,?)", phases)
con.executemany("INSERT INTO event_types VALUES (?,?,?,?)", event_types)
con.commit()
print(f"Seeded {len(phases)} lifecycle phases and {len(event_types)} event types.")
for et in con.execute("SELECT event_type_code, phase_code, is_completion_signal FROM event_types"):
    print("  ", et)

### What just happened
The vocabulary is seeded: eight abstract phases and the four `permit_*` event types JN-A will
emit. Note `permit_finaled` and `permit_completed` carry `is_completion_signal=1` — that flag is
what later notebooks fold to count completions, and it's why finaled and issued must be *separate*
event types (the Table A2 CO section reads finaled; the BP section reads issued). Everything here
is city-neutral on purpose: another city's adapter maps its own event labels into these same codes.

### What this cell does
This is the **conservation baseline**. Before emitting a single event, we read each feed file,
record exactly how many data rows it has *per file* (stating our row-count rule explicitly), and
register the source + an ingestion run. The `rows_in_source` number captured here is the
denominator the final proof checks against — so it must be counted *before* any cleaning, and the
rule for what counts as a row must be stated, not assumed.

In [ ]:
import hashlib
feed_files = sorted(FEED_DIR.glob(FEED_GLOB))
assert feed_files, f"No feed files matched {FEED_GLOB} in {FEED_DIR}"

per_file = []
frames   = []
for f in feed_files:
    # ROW-COUNT RULE: read with the known header row; a "data row" = any row pandas
    # returns below the header. We do NOT drop blanks here - we count them and handle
    # them explicitly at the checkpoint (a row with all-null dates is represented, not dropped).
    df = pd.read_excel(f, header=HEADER_ROW, dtype=str)  # dtype=str: preserve verbatim, parse later
    df["_source_file"] = f.name
    n = len(df)
    per_file.append((f.name, n, round(f.stat().st_size/1e6, 2)))
    frames.append(df)
    # register the source row
    chk = hashlib.sha256(f.read_bytes()).hexdigest()[:16]
    con.execute(
        "INSERT INTO sources (source_kind, city, locator, retrieved_at, checksum, notes) "
        "VALUES ('cpra_permit_feed','Berkeley',?,?,?,?)",
        (str(f), NOW, chk, "CPRA BP Annual Permit Report"))

raw = pd.concat(frames, ignore_index=True)
rows_in_source = len(raw)

print("Per-file data-row counts (the conservation denominator):")
for name, n, mb in per_file:
    print(f"  {name:<45} {n:>7,} rows   ({mb} MB)")
print(f"  {'TOTAL':<45} {rows_in_source:>7,} rows")
print()
print(f"Expected ~{EXPECT_RAW_ROWS:,} (probe). Match: {rows_in_source == EXPECT_RAW_ROWS}")

source_ids = [r[0] for r in con.execute("SELECT source_id FROM sources")]
con.execute(
    "INSERT INTO ingestion_runs (source_id, started_at, rows_in_source, rows_ingested, "
    "rows_rejected, conserved) VALUES (?,?,?,0,0,0)", (source_ids[0], NOW, rows_in_source))
RUN_ID = con.execute("SELECT MAX(run_id) FROM ingestion_runs").fetchone()[0]
con.commit()
print(f"\nIngestion run #{RUN_ID} opened. rows_in_source = {rows_in_source:,}")

### What just happened
We now have the **denominator** for the whole notebook: the exact per-file and total data-row
counts, captured before any transformation, with the row-count rule stated (every row pandas
returns below the known header, blanks included). If the total doesn't match the probe's 32,202,
that's a signal the source files changed since the probe — investigate before proceeding. Both
files are registered in `sources` with checksums, and an `ingestion_runs` row is open with
`rows_ingested` still 0. Note we read everything as **strings** (`dtype=str`) so values are
preserved verbatim — including the REV cumulative-restatement UnitsAdded we must *not* fix here.

### What this cell does
The heart of JN-A. For **every** source row, for **every** of the four lifecycle date fields that
is *populated*, emit one event. A row with Submittal + Issuance + Finaled becomes **three** events,
all sharing `source_record_key = PermitNumber`. The full original row is preserved verbatim in
`raw_payload` (as JSON) so any later re-classification reads from intact evidence. We parse dates
tolerantly and record precision honestly. **No row is filtered. No date is invented. No dedup.**
We also tally, per date field, how many events we emit — those tallies are the per-axis
conservation anchors.

In [ ]:
def parse_date(val):
    "Tolerant parse to (iso_date, precision). Mixed formats per the probe."
    if val is None or (isinstance(val, float)) or str(val).strip() in ("", "nan", "NaT"):
        return None, "unknown"
    s = str(val).strip()
    for fmt, prec in (("%Y-%m-%d %H:%M:%S","day"), ("%Y-%m-%d","day"),
                      ("%m/%d/%Y","day"), ("%m/%d/%y","day")):
        try:
            return dt.datetime.strptime(s, fmt).date().isoformat(), prec
        except ValueError:
            continue
    # last resort: let pandas try, but flag precision as uncertain
    try:
        d = pd.to_datetime(s, errors="raise").date().isoformat()
        return d, "day"
    except Exception:
        return None, "unparseable"

event_rows   = []          # tuples for executemany into events
per_axis      = {c: 0 for c in DATE_EVENT_MAP}
# ROW accounting (mutually exclusive - every source row lands in exactly ONE):
rows_with_event = 0        # >=1 event emitted
rows_no_event   = 0        # 0 events emitted (all dates null and/or all unparseable)
# FIELD-level data-quality stat (NOT a row bucket - a row can have both good and bad fields):
rejected_fields = 0        # individual date cells that were present but unparseable
rejected_detail = []
in_win = out_win = undated = 0
W0, W1 = REQUEST_WINDOW

for idx, row in raw.iterrows():
    payload = json.dumps({k: (None if pd.isna(v) else v) for k, v in row.items()},
                         ensure_ascii=False)
    permit  = row.get(COL_PERMIT)
    permit  = None if pd.isna(permit) else str(permit).strip()
    addr    = row.get("Address") if "Address" in raw.columns else None
    apn     = row.get("APN") if "APN" in raw.columns else None
    desc    = row.get("Description") if "Description" in raw.columns else None
    units   = row.get("UnitsAdded") if "UnitsAdded" in raw.columns else None
    src_file = row["_source_file"]
    sid = source_ids[0] if "2018-2022" in src_file else source_ids[-1]

    emitted_this_row = 0
    for col, (etype, _is_co) in DATE_EVENT_MAP.items():
        if col not in raw.columns:
            continue
        iso, prec = parse_date(row.get(col))
        if iso is None and prec in ("unknown",):
            continue  # field genuinely empty - no event for this field (never invent)
        if iso is None and prec == "unparseable":
            rejected_detail.append({"permit": permit, "field": col, "raw": str(row.get(col))})
            rejected_fields += 1   # a FIELD-level data-quality count, not a row bucket
            continue
        # window flag (NEVER used to filter - recorded only)
        d = dt.date.fromisoformat(iso)
        win = "in" if W0 <= d <= W1 else "out"
        if win == "in": in_win += 1
        else: out_win += 1
        event_rows.append((etype, iso, prec, permit, int(sid), RUN_ID, payload,
                           addr if not pd.isna(addr) else None,
                           apn if not pd.isna(apn) else None,
                           desc if not pd.isna(desc) else None,
                           None if (units is None or pd.isna(units)) else str(units),
                           NOW))
        per_axis[col] += 1
        emitted_this_row += 1

    if emitted_this_row == 0:
        rows_no_event += 1     # all dates null and/or all unparseable - row represented as zero-event
    else:
        rows_with_event += 1   # >=1 event; may ALSO have had an unparseable field (counted separately)

con.executemany(
    "INSERT INTO events (event_type_code, event_date, event_date_precision, source_record_key, "
    "source_id, ingestion_run_id, raw_payload, raw_address, raw_apn, raw_description, raw_units, "
    "created_at) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)", event_rows)
con.commit()

print(f"Emitted {len(event_rows):,} events from {rows_in_source:,} source rows.")
print(f"  rows with >=1 event : {rows_with_event:,}")
print(f"  rows with 0 events  : {rows_no_event:,}  (all dates null/unparseable - represented, NOT dropped)")
print(f"  unparseable FIELDS  : {rejected_fields:,}  (individual bad date cells - logged; a row with one")
print(f"                         can still have valid dates and be counted above)")
print()
print("Per-axis event tallies vs probe anchors:")
for col,(etype,_) in DATE_EVENT_MAP.items():
    anchor = {COL_SUBMITTAL:EXPECT_SUBMITTAL, COL_ISSUANCE:EXPECT_ISSUANCE,
              COL_FINALED:EXPECT_FINALED, COL_COMPLETED:EXPECT_COMPLETED}[col]
    flag = "OK" if per_axis[col] == anchor else "** MISMATCH"
    print(f"  {etype:<18} {per_axis[col]:>7,}   (probe {anchor:>6,})  {flag}")
print()
print(f"Window flag (recorded, never filtered): in-window {in_win:,} - out-of-window {out_win:,}")
if rejected_detail:
    print("\nRejected (unparseable) - never silent:")
    for r in rejected_detail[:20]:
        print("  ", r)

### What just happened
Every source row has been exploded into its constituent lifecycle events. The decisive output is
the **per-axis tally vs the probe anchors**: the count of `permit_submitted` events must equal
32,202, `permit_issued` 31,940, `permit_finaled` 21,650, `permit_completed` 1. These are four
*independent* ground-truth checks — passing all four is far stronger evidence of fidelity than a
single total would be. Any `‼ MISMATCH` means that specific date axis dropped or duplicated rows,
and you can see exactly which one. The window flag confirms how many dates fall outside the
2018–2025 request — recorded as a fact, never used to filter. Any unparseable date is logged with
its permit and raw value, never silently discarded.

### What this cell does
The **single guarded boundary**. This cell proves — arithmetically, against the database — that
nothing vanished. It performs two conservation checks and writes the verdict back to
`ingestion_runs`. Crucially, it emits the checks as **standalone SQL/Python you can re-run
yourself** against `berkeley_housing_v4.db`, so you are not trusting the notebook's report — you
are verifying it. This is the cell that replaces the entire forensic-audit era: in v1→v2→v3 a drop
produced no failure, just a smaller number nobody could explain. Here a drop trips an assertion
at the exact boundary it occurred.

In [ ]:
events_total = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]

# CHECK 1 - ROW conservation: every source row is represented (mutually exclusive buckets).
#   rows_in_source == rows_with_event + rows_no_event
#   (rejected_fields is a FIELD-level data-quality stat, NOT a row bucket - kept out of this identity)
row_lhs = rows_in_source
row_rhs = rows_with_event + rows_no_event
row_ok  = (row_lhs == row_rhs)

# CHECK 2 - EVENT conservation: total events == sum of present, parseable date fields.
axis_sum = sum(per_axis.values())
evt_ok   = (axis_sum == events_total)

# CHECK 3 - dedup floor: events must comfortably exceed the dedup floor (else something deduped).
floor_ok = events_total > EXPECT_DEDUP_FLOOR

conserved = 1 if (row_ok and evt_ok and floor_ok) else 0
con.execute("UPDATE ingestion_runs SET rows_ingested=?, rows_rejected=?, conserved=?, "
            "rejected_detail=? WHERE run_id=?",
            (events_total, rejected_fields, conserved,
             json.dumps(rejected_detail) if rejected_detail else None, RUN_ID))
con.commit()

print("CONSERVATION CHECKPOINT")
print("="*64)
print(f"CHECK 1  ROW conservation:   {row_lhs:,} == {row_rhs:,}   -> {'PASS' if row_ok else 'FAIL'}")
print(f"         ({rows_with_event:,} with-event + {rows_no_event:,} no-event; every row in exactly one bucket)")
print(f"CHECK 2  EVENT conservation: sum(axes) {axis_sum:,} == events {events_total:,} -> {'PASS' if evt_ok else 'FAIL'}")
print(f"CHECK 3  dedup floor:        events {events_total:,} > {EXPECT_DEDUP_FLOOR:,} -> {'PASS' if floor_ok else 'FAIL'}")
print(f"         data-quality (not a conservation failure): {rejected_fields:,} unparseable date FIELDS logged")
print("="*64)
print(f"ingestion_runs.conserved = {conserved}   ({'CONSERVED' if conserved else 'NOT CONSERVED - STOP'})")
assert conserved == 1, "CONSERVATION FAILED - do not proceed; data was lost or invented at ingest."

### What just happened
The three conservation checks passed and `ingestion_runs.conserved` is set to 1. **CHECK 1** proves
every source row is represented in exactly one bucket — emitted at least one event, or emitted none
(all its dates null or unparseable) — with no remainder and no double-counting. **CHECK 2** proves the
events table holds exactly as many rows as there were populated, parseable date fields. **CHECK 3**
confirms the event count exceeds the dedup floor, catching the one bug that would matter here (an
accidental dedup). Unparseable date *fields* are reported as a data-quality stat, deliberately kept
out of the row-conservation identity — a row with one bad date can still contribute its good dates and
remains fully represented. The final `assert` is the hard stop: if conservation had failed, the
notebook halts rather than letting a lossy ingest flow downstream.

### What this cell does
Emits a **standalone verification script** — pure SQL plus the source-row count — that you (John)
can run against `berkeley_housing_v4.db` independently of this notebook, at any time, to re-confirm
the conservation result. The discipline is *verify, don't trust*: the notebook's PASS is only
believable if you can reproduce it with your own hands. This cell writes that script to disk.

In [ ]:
verify = f'''#!/usr/bin/env python3
# Re-run the JN-A conservation check independently. Read-only on v4.db.
import sqlite3, glob, pandas as pd
from pathlib import Path

DB   = "{DB_PATH}"
FEED = sorted(glob.glob("{FEED_DIR}/{FEED_GLOB}"))

# 1. recount source rows straight from the xlsx (the independent denominator)
src = sum(len(pd.read_excel(f, header={HEADER_ROW}, dtype=str)) for f in FEED)

con = sqlite3.connect(DB)
events = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
by_type = dict(con.execute(
    "SELECT event_type_code, COUNT(*) FROM events GROUP BY event_type_code").fetchall())
run = con.execute("SELECT rows_in_source, rows_ingested, rows_rejected, conserved "
                  "FROM ingestion_runs ORDER BY run_id DESC LIMIT 1").fetchone()

print("source rows recounted from xlsx :", src)
print("events in db                    :", events)
print("events by type                  :", by_type)
print("ingestion_runs (in/ingested/rej/conserved):", run)
print()
print("ANCHORS  submitted=32202 issued=31940 finaled=21650 completed=1")
ok = (by_type.get("permit_submitted")==32202 and by_type.get("permit_issued")==31940
      and by_type.get("permit_finaled")==21650 and run[3]==1)
print("INDEPENDENT VERDICT:", "PASS" if ok else "FAIL - investigate")
'''
out = REPO / "scripts" / "verify_jn_a_conservation.py"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(verify)
print(f"Wrote re-runnable verifier -> {out}")
print("Run it yourself any time with:  python3", out)

### What just happened
A standalone verifier now lives at `scripts/verify_jn_a_conservation.py`. It recounts the source
rows directly from the xlsx (an independent denominator, not the notebook's cached number),
reads the event counts from the database, and re-checks the four anchors and the conserved flag.
Run it whenever you want to confirm — without this notebook in the loop — that the ingest is still
faithful. That independence is the point: the conservation claim is only as trustworthy as your
ability to reproduce it.

### What this cell does
Reports the descriptive facts of the ingested stream — *facts only, no judgement*. Event-type
distribution, the all-null and duplicate-key statistics, the in/out-of-window split, and a sample
of one permit's exploded events showing its preserved `raw_payload`. None of this classifies or
filters anything; it is the honest picture of what now sits in the event stream, ready for JN-B
(typing refinement) and JN-C (the reversible classifier, where the phantom-master work lands).

In [ ]:
print("EVENT-TYPE DISTRIBUTION")
for r in con.execute("SELECT event_type_code, COUNT(*) FROM events GROUP BY event_type_code ORDER BY 2 DESC"):
    print(f"  {r[0]:<18} {r[1]:>7,}")

print("\nDUPLICATE source_record_key (cross-file overlap - REPORTED, not resolved)")
dup = con.execute(
    "SELECT COUNT(*) FROM (SELECT source_record_key FROM events "
    "WHERE source_record_key IS NOT NULL GROUP BY source_record_key HAVING COUNT(DISTINCT raw_payload) > 1)"
).fetchone()[0]
maxmult = con.execute(
    "SELECT MAX(c) FROM (SELECT COUNT(*) c FROM events GROUP BY source_record_key)").fetchone()[0]
print(f"  permit numbers appearing in >1 distinct source row: {dup:,}")
print(f"  max events sharing one permit number              : {maxmult}")

print("\nWINDOW SPLIT (recorded fact, never a filter)")
print(f"  in-window {in_win:,} - out-of-window {out_win:,}")

print("\nSAMPLE - one permit's exploded events (raw_payload preserved):")
sample_key = con.execute(
    "SELECT source_record_key FROM events WHERE source_record_key IS NOT NULL "
    "GROUP BY source_record_key HAVING COUNT(*) >= 3 LIMIT 1").fetchone()[0]
for r in con.execute(
    "SELECT event_id, event_type_code, event_date, event_date_precision "
    "FROM events WHERE source_record_key=? ORDER BY event_date", (sample_key,)):
    print("  ", r)
print(f"  (permit {sample_key} - its events span its lifecycle, keyed by permit not address)")

### What just happened
This is the honest portrait of the ingested stream. The event-type distribution should mirror the
per-axis anchors; the duplicate-key count is the cross-file overlap (~1,430 permits appear in both
annual reports) — **preserved as multiple events, to be resolved later as a label, never dropped
here**. The sample shows a single permit exploded into its lifecycle events, keyed by permit number
rather than address — which is the whole architectural point: this permit's submittal, issuance, and
finaling are now three dated, sourced events on the spine, and *no* projection has yet decided what
building they belong to. That decision is JN-D's, made by folding this intact evidence.

---
## JN-A complete — what now exists, and what comes next

**What now exists:** a fresh `berkeley_housing_v4.db` whose `events` table holds every lifecycle
event from every row the city sent, each preserving its full source row verbatim, each traceable to
a registered source, with a **proven, re-runnable conservation guarantee** that nothing vanished at
the ingest boundary.

**What JN-A deliberately did NOT do:** classify housing vs non-housing, identify master permits,
dedup the cross-file overlap, resolve the REV/UnitsAdded restatement, or project any structures.
All of that is downstream, and all of it operates as **reversible labels or re-runnable projections
over this permanent evidence** — never as deletions.

**The discipline this notebook establishes for the whole pipeline:** every stage ends with a
conservation checkpoint that asserts it lost nothing, checked against the stage before it. That is
how silent data-loss — the failure that defined v1→v2→v3 — is designed out: not by being careful,
but by making loss trip an assertion at the exact stage it occurs.

**Next:** JN-B refines event typing from the raw fields (issuance vs finaled vs the rest, precisely),
then JN-C applies the reversible housing/master/phantom classifier — where the phantom-master
prose-accuracy work finally lands, on intact evidence. Neither can lose data; both are re-runnable
over the events JN-A proved complete.

*Gated discipline reminder: this notebook writes only to `berkeley_housing_v4.db` (a fresh build),
never to v3, and commits nothing. Run it, read the checkpoint, run the standalone verifier yourself,
and only then decide what to commit.*